In [ ]:
# =============================================================================
# Zelle 01 – Setup: Pfade, Konfiguration, Stil, Datenbasis prüfen
# =============================================================================
# Zweck: Gemeinsame Grundlage für alle EDA-Zellen.
# Datenbasis: ausschließlich der Snapshot in data/raw (Prinzip P7).
#   Einzige dokumentierte Ausnahme: Zelle 10 liest Tags nur lesend vom Stick
#   (Entscheidung 07.11.2026; 126 GB Kopie unverhältnismäßig).
# Artefakte: Namensschema 02_<Zelle>_<slug>; Tabellen über tabelle_speichern(),
#   Grafiken über figur_speichern() – beide prüfen das Schema.
# =============================================================================

import sys
from pathlib import Path

import pandas as pd
import yaml

from playlist_agent.viz_config import ARTEFAKT_MUSTER, FARBEN, figur_speichern, stil_aktivieren

# --- Projektpfade (gleiche Logik wie Notebook 01) ----------------------------
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert PROJECT_ROOT.name == "_MyMSIT_AI-Agent_repair-playlist", PROJECT_ROOT

RAW = PROJECT_ROOT / "data" / "raw"
FIG = PROJECT_ROOT / "reports" / "figures"
TAB = PROJECT_ROOT / "reports" / "tables"
ERR = PROJECT_ROOT / "reports" / "errors"      # privat (gitignored)

cfg = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
stil_aktivieren()


def tabelle_speichern(df: pd.DataFrame, name: str, ordner: Path = TAB) -> Path:
    """CSV nach dem Namensschema <NB>_<Zelle>_<slug> speichern.

    Gleiche Prüfung wie figur_speichern(). Vorerst hier im Notebook;
    Umzug nach src/ in Phase 3, sobald sie mehrfach gebraucht wird.
    """
    if not ARTEFAKT_MUSTER.match(name):
        raise ValueError(f"Name '{name}' entspricht nicht <NB>_<Zelle>_<slug>.")
    pfad = ordner / f"{name}.csv"
    df.to_csv(pfad, index=False, encoding="utf-8")
    return pfad


# --- Datenbasis prüfen: Snapshot vollständig? ---------------------------------
df_manifest = pd.read_csv(RAW / "snapshot_manifest.csv")
df_index = pd.read_csv(RAW / "index" / "musik_index.csv")

print(f"Python  : {sys.version.split()[0]}")
print(f"pandas  : {pd.__version__}")
print(f"Snapshot: {len(df_manifest)} Dateien, davon bitgenau {df_manifest['identisch'].sum()}")
print(df_manifest.groupby("quelle").size().to_string())
print(f"Index   : {len(df_index)} Dateien")
print(f"\nDatentyp Spalte 'pfad_relativ' im Index: {df_index['pfad_relativ'].dtype}")

In [ ]:
# =============================================================================
# Zelle 02 – Alle Crates parsen: Einträge, Verteilung, Integrität, pandas-3-Check
# =============================================================================
# Zweck: Vollständige Tabelle aller Crate-Einträge als Grundlage der EDA.
# Format (Vorbefund): TLV-Blöcke, 4-Byte-Tag + 4-Byte-Länge Big-Endian,
#   Strings UTF-16-BE; Track = 'otrk', darin Pfad = 'ptrk'.
# Integritätsprüfung: Datei muss exakt bis zum letzten Byte aufgehen.
# pandas-3-Check: bleibt die Unicode-Normalform (NFC/NFD) in Spalten erhalten?
# Datenschutz: Einzeltabelle mit Crate-Namen/Pfaden privat in data/interim;
#   ins Repo nur Verteilungsstatistik und Grafik.
# Prototyp im Notebook, Umzug nach src/playlist_agent/parsers/ in Phase 3.
# =============================================================================

import unicodedata
import matplotlib.pyplot as plt


def parse_crate(pfad: Path) -> dict:
    """Liest eine Serato-Crate; liefert Version, Pfadliste und Integritätsstatus."""
    b = pfad.read_bytes()
    i, version, pfade = 0, None, []
    while i + 8 <= len(b):
        tag, n = b[i:i + 4], int.from_bytes(b[i + 4:i + 8], "big")
        wert = b[i + 8:i + 8 + n]
        i += 8 + n
        if tag == b"vrsn":
            version = wert.decode("utf-16-be")
        elif tag == b"otrk":                      # Track-Block: darin 'ptrk' suchen
            j = 0
            while j + 8 <= len(wert):
                t2, n2 = wert[j:j + 4], int.from_bytes(wert[j + 4:j + 8], "big")
                if t2 == b"ptrk":
                    pfade.append(wert[j + 8:j + 8 + n2].decode("utf-16-be"))
                j += 8 + n2
    return {"version": version, "pfade": pfade, "vollstaendig": i == len(b)}


# --- Alle Crates parsen ---------------------------------------------------------
crate_dateien = sorted((RAW / "crates").glob("*.crate"))
eintraege, crates = [], []
for f in crate_dateien:
    try:
        r = parse_crate(f)
        crates.append({"crate": f.stem, "version": r["version"],
                       "n_tracks": len(r["pfade"]), "vollstaendig": r["vollstaendig"],
                       "fehler": None})
        eintraege += [{"crate": f.stem, "position": k, "pfad": p}
                      for k, p in enumerate(r["pfade"], start=1)]
    except Exception as e:                        # einzelne Crate darf nicht alles stoppen
        crates.append({"crate": f.stem, "version": None, "n_tracks": 0,
                       "vollstaendig": False, "fehler": repr(e)})

df_crates = pd.DataFrame(crates)                  # 1 Zeile je Crate
df_eintraege = pd.DataFrame(eintraege)            # 1 Zeile je Crate-Eintrag
df_eintraege.to_csv(PROJECT_ROOT / "data/interim/02_02_crate_eintraege.csv",
                    index=False, encoding="utf-8")   # privat

# --- Integrität -------------------------------------------------------------------
print(f"Crates gelesen       : {len(df_crates)}")
print(f"Versionen            : {df_crates['version'].value_counts(dropna=False).to_dict()}")
print(f"vollständig gelesen  : {df_crates['vollstaendig'].sum()} / {len(df_crates)}")
print(f"Lesefehler           : {df_crates['fehler'].notna().sum()}")
print(f"leere Crates (0 Tr.) : {(df_crates['n_tracks'] == 0).sum()}")

# --- Umfang ------------------------------------------------------------------------
print(f"\nEinträge gesamt      : {len(df_eintraege)}")
print(f"eindeutige Pfade     : {df_eintraege['pfad'].nunique()}")
stat_tracks = df_crates["n_tracks"].describe(percentiles=[.25, .5, .75, .9]).round(1)
print("\nTracks je Crate:\n" + stat_tracks.to_string())

# --- pandas-3-Check: Normalform bleibt erhalten? -----------------------------------
n_nfd_python = sum(unicodedata.normalize("NFC", p) != p for p in df_eintraege["pfad"].tolist())
n_nfd_pandas = (df_eintraege["pfad"] != df_eintraege["pfad"].str.normalize("NFC")).sum()
print(f"\nNFD-Pfade (reines Python): {n_nfd_python} | (pandas-Vergleich): {n_nfd_pandas} "
      f"| Datentyp: {df_eintraege['pfad'].dtype}")

# --- Artefakt 1: Verteilungsstatistik (public, ohne Namen) --------------------------
df_tracks_je_crate = (stat_tracks.rename("tracks_je_crate").reset_index()
                      .rename(columns={"index": "kennzahl"}))
tabelle_speichern(df_tracks_je_crate, "02_02_tracks_je_crate")

# --- Artefakt 2: Histogramm (log-Achse wegen starker Rechtsschiefe) ---------------
# Begründung: Median 13 vs. Max 1330 -> lineare Achse drückt 140 von 167 Crates
#   in den ersten Balken. Log-Klassen geben jeder Größenordnung gleich viel Platz.
import numpy as np

klassen = np.logspace(0, np.log10(df_crates["n_tracks"].max()), 25)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_crates["n_tracks"], bins=klassen, color=FARBEN["gold"],
        edgecolor=FARBEN["hintergrund"])
ax.axvline(df_crates["n_tracks"].median(), color=FARBEN["blau"], lw=2,
           label=f"Median = {df_crates['n_tracks'].median():.0f}")
ax.set_xscale("log")
ax.set_xticks([1, 3, 10, 30, 100, 300, 1000])
ax.set_xticklabels(["1", "3", "10", "30", "100", "300", "1000"])
ax.set_xlabel("Tracks je Crate (logarithmische Achse)")
ax.set_ylabel("Anzahl Crates")
ax.set_title(f"Größenverteilung der {len(df_crates)} Serato-Crates")
ax.legend()
figur_speichern(fig, "02_02_tracks_je_crate", FIG)
plt.show()

In [ ]:
# =============================================================================
# Zelle 03 – Zeichenkodierung der Crate-Pfade: Fallstudie, Klassen, Wirkung, je Crate
# =============================================================================
# Anlass: Stichprobe "Coming Home" – Crate zeigt im Editor "V a t h", Stick "Väth".
#   Lücken = UTF-16-Darstellung im Editor (00-Bytes), keine echten Leerzeichen.
#   Offen: Ist das "a" + kombinierendes Trema (NFD) oder ein verlorener Umlaut?
# Methode:
#   A Fallstudie Codepoints | B Klassen ASCII/NFC/NFD je eindeutigem Pfad
#   C Treffer im Index: exakt -> erst nach NFC -> erst nach Akzent-Entfernung
#   D NFD je Crate (Prüfung der auffälligen Zahl 656)
# Datenschutz: Zählstatistik public (reports/tables), Einzelpfade privat (reports/errors).
# =============================================================================


def akzente_entfernen(s: str) -> str:
    """'Väth' -> 'vath': zerlegt Zeichen (NFKD), entfernt Akzente, Kleinschreibung."""
    return "".join(c for c in unicodedata.normalize("NFKD", s)
                   if not unicodedata.combining(c)).lower()


def klasse(p: str) -> str:
    if all(ord(c) < 128 for c in p):
        return "ASCII"
    return "NFD" if unicodedata.normalize("NFC", p) != p else "NFC mit Sonderzeichen"


# --- A: Fallstudie "Coming Home" ----------------------------------------------------
print("=== A: Fallstudie 'Coming Home' ===")
for quelle, serie in [("Crate", df_eintraege["pfad"]), ("Index", df_index["pfad_relativ"])]:
    for p in serie[serie.str.contains("Coming Home", regex=False)].unique():
        zeichen = [f"'{c}' U+{ord(c):04X}" for c in p if ord(c) > 127]
        print(f"{quelle}: {p!r}\n   Sonderzeichen: {zeichen} | Klasse: {klasse(p)}")

# --- B + C: Klassen und Treffer je eindeutigem Crate-Pfad ---------------------------
idx_exakt = set(df_index["pfad_relativ"])
idx_nfc = {unicodedata.normalize("NFC", p) for p in idx_exakt}
idx_akzentfrei = {akzente_entfernen(p) for p in idx_exakt}

df_pfade = pd.DataFrame({"pfad": df_eintraege["pfad"].unique()})
df_pfade["klasse"] = df_pfade["pfad"].map(klasse)
df_pfade["treffer"] = np.select(
    [df_pfade["pfad"].isin(idx_exakt),
     df_pfade["pfad"].map(lambda p: unicodedata.normalize("NFC", p) in idx_nfc),
     df_pfade["pfad"].map(lambda p: akzente_entfernen(p) in idx_akzentfrei)],
    ["1 exakt", "2 erst nach NFC", "3 erst ohne Akzente"],
    default="4 kein Treffer")

df_klasse_treffer = pd.crosstab(df_pfade["klasse"], df_pfade["treffer"],
                                margins=True, margins_name="Summe")
print("\n=== B+C: eindeutige Crate-Pfade nach Klasse und Treffer im Index ===")
print(df_klasse_treffer.to_string())
tabelle_speichern(df_klasse_treffer.reset_index(), "02_03_kodierung_klasse_treffer")

# Einzelpfade der schwierigen Fälle privat sichern (für Zelle 05)
df_pfade[df_pfade["treffer"].isin(["3 erst ohne Akzente", "4 kein Treffer"])] \
    .to_csv(ERR / "02_03_pfade_ohne_direkten_treffer.csv", index=False, encoding="utf-8")

# --- D: NFD je Crate ----------------------------------------------------------------
df_eintraege["nfd"] = df_eintraege["pfad"].map(lambda p: klasse(p) == "NFD")
df_nfd_je_crate = (df_eintraege.groupby("crate")["nfd"]
                   .agg(n_eintraege="size", n_nfd="sum").reset_index())
n_crates_mit_nfd = (df_nfd_je_crate["n_nfd"] > 0).sum()
print(f"\n=== D: NFD je Crate ===")
print(f"NFD-Einträge gesamt     : {df_eintraege['nfd'].sum()}  "
      f"(eindeutige NFD-Pfade: {(df_pfade['klasse'] == 'NFD').sum()})")
print(f"Crates mit NFD-Einträgen: {n_crates_mit_nfd} von {len(df_nfd_je_crate)}")
print("Top 5 Crates nach NFD-Einträgen:")
print(df_nfd_je_crate.sort_values("n_nfd", ascending=False).head(5).to_string(index=False))

# --- Grafik: Treffer je Klasse (eindeutige Pfade) ------------------------------------
# Farbbedeutung: Grün = direkt gelöst, Gold = Kernbefund (erst nach NFC),
#   Blau = Akzent verloren, Grau = kein Treffer.
df_plot = df_klasse_treffer.drop(index="Summe", columns="Summe")
farben = {"1 exakt": FARBEN["gruen"], "2 erst nach NFC": FARBEN["gold"],
          "3 erst ohne Akzente": FARBEN["blau"], "4 kein Treffer": FARBEN["text_gedimmt"]}
fig, ax = plt.subplots(figsize=(10, 3.5))
links = np.zeros(len(df_plot))
for spalte in df_plot.columns:
    ax.barh(df_plot.index, df_plot[spalte], left=links, color=farben[spalte], label=spalte)
    links += df_plot[spalte].to_numpy()
ax.set_xlabel("eindeutige Crate-Pfade")
ax.set_title("Crate-Pfade nach Zeichenkodierung und Treffer im Index")
ax.legend(loc="lower right", fontsize=10)
figur_speichern(fig, "02_03_kodierung_klasse_treffer", FIG)
plt.show()

In [ ]:
# =============================================================================
# Zelle 04 – Evidenz der Trefferstufen: Index-Format, Beispiele je Stufe,
#            Stichprobe für unabhängigen foobar-Test
# =============================================================================
# Zweck: Behauptungen aus Zelle 03 an konkreten Pfaden belegen (erst Evidenz,
#   dann Aussage) und eine Stichprobe für eine UNABHÄNGIGE Referenz erzeugen:
#   Songs der Stichprobe werden von Hand in foobar eingefügt und als Playlist
#   gespeichert; der von foobar geschriebene Pfad wird in Zelle 05 mit Index
#   und Crate verglichen.
# Reproduzierbar: Zufallsauswahl mit festem Startwert (SEED = 42).
# Datenschutz: Pfade nur als Notebook-Ausgabe (nbstripout entfernt sie beim
#   Commit); Stichprobenliste privat in data/gold/.
# =============================================================================

SEED = 42
STICK_LW = "D:"          # Laufwerk des Sticks auf dem Asus (Zelle 06, Notebook 01)


def sonderzeichen(p: str) -> list[str]:
    """Alle Nicht-ASCII-Zeichen mit Codepoint, z. B. ['ä' U+00E4]."""
    return [f"'{c}' U+{ord(c):04X}" for c in p if ord(c) > 127]


def als_windows_pfad(p: str) -> str:
    """'Musik/A/x.mp3' -> 'D:\\Musik\\A\\x.mp3' (zum Suchen im Explorer/foobar)."""
    return STICK_LW + "\\" + p.replace("/", "\\")


# Zuordnung NFC-Form -> tatsächlicher Index-Pfad (für Stufe 2: welcher Pfad rechts?)
idx_nfc_zu_pfad = {unicodedata.normalize("NFC", p): p for p in df_index["pfad_relativ"]}


def index_partner(p: str, stufe: str) -> str | None:
    if stufe == "1 exakt":
        return p
    if stufe == "2 erst nach NFC":
        return idx_nfc_zu_pfad.get(unicodedata.normalize("NFC", p))
    return None


def zeige(links: str, stufe: str) -> None:
    rechts = index_partner(links, stufe)
    print(f"  Stufe : {stufe}")
    print(f"  links (Crate): {links!r}")
    print(f"        Sonderzeichen: {sonderzeichen(links)}")
    print(f"  rechts (Index): {rechts!r}")
    if rechts:
        print(f"        Sonderzeichen: {sonderzeichen(rechts)}")
        print(f"        Zeichen-identisch: {links == rechts}")
    print()


# --- A: Format eines Index-Pfads ("rechts") ----------------------------------------
print("=== A: So sieht ein Pfad im Index aus (3 Zufallsbeispiele) ===")
for p in df_index["pfad_relativ"].sample(3, random_state=SEED):
    print(f"  {p!r}")
print("  Format: relativ zur Stick-Wurzel, ohne Laufwerk, Trenner '/', "
      "gleiche Form wie in den Crates.\n")

# --- B: Nutzer-Stichprobe (ohne Umlaut): Suche in Crates UND Index ---------------
print("=== B: Beispiele aus der Nutzer-Stichprobe ===")
for suchtext in ["08 Skorbut (I-Cube Remix)", "08 I'm Ready- Voices and Atmosphere"]:
    funde_crate = df_pfade[df_pfade["pfad"].str.contains(suchtext, regex=False)]
    funde_index = df_index[df_index["pfad_relativ"].str.contains(suchtext, regex=False)]
    print(f"Suche '{suchtext}': {len(funde_crate)} Crate-Pfad(e), "
          f"{len(funde_index)} Index-Pfad(e)")
    for _, z in funde_crate.iterrows():
        zeige(z["pfad"], z["treffer"])
    if funde_crate.empty:
        for p in funde_index["pfad_relativ"]:
            print(f"  nur im Index (in keiner Crate): {p!r}\n")

# --- C: je Stufe 3 Zufallsbeispiele ---------------------------------------------------
print("=== C: je Trefferstufe 3 Zufallsbeispiele ===")
for stufe in ["1 exakt", "2 erst nach NFC", "3 erst ohne Akzente", "4 kein Treffer"]:
    gruppe = df_pfade[df_pfade["treffer"] == stufe]
    print(f"--- {stufe}: {len(gruppe)} Pfade insgesamt ---")
    if gruppe.empty:
        print("  (keine Fälle – Befund, kein fehlendes Beispiel)\n")
        continue
    for p in gruppe["pfad"].sample(min(3, len(gruppe)), random_state=SEED):
        zeige(p, stufe)


# --- D: Stichprobe für den foobar-Test ----------------------------------------------
# 4 x Stufe 1 ASCII, 3 x Stufe 1 mit Sonderzeichen, 3 x Stufe 2 (NFD),
# 3 x Stufe 4 (Titel wird von Hand gesucht: gefunden -> in foobar aufnehmen,
#   nicht gefunden -> als "fehlt" notieren).
plan = [("1 exakt", "ASCII", 4), ("1 exakt", "NFC mit Sonderzeichen", 3),
        ("2 erst nach NFC", "NFD", 3), ("4 kein Treffer", None, 3)]
teile = []
for stufe, kl, n in plan:
    g = df_pfade[(df_pfade["treffer"] == stufe) & ((df_pfade["klasse"] == kl) if kl else True)]
    teile.append(g.sample(n, random_state=SEED))
df_stichprobe = pd.concat(teile, ignore_index=True)
df_stichprobe["index_pfad"] = [index_partner(p, s) for p, s in
                               zip(df_stichprobe["pfad"], df_stichprobe["treffer"])]
df_stichprobe["suchen_unter"] = [als_windows_pfad(ip) if pd.notna(ip) else
                                 "TITEL VON HAND SUCHEN: " + p.rsplit("/", 1)[-1]
                                 for ip, p in zip(df_stichprobe["index_pfad"],
                                                  df_stichprobe["pfad"])]
df_stichprobe.insert(0, "nr", range(1, len(df_stichprobe) + 1))
df_stichprobe.to_csv(PROJECT_ROOT / "data/gold/02_04_stichprobe_foobar.csv",
                     index=False, encoding="utf-8")       # privat

print("=== D: Stichprobe für den foobar-Test (gespeichert: data/gold/02_04_stichprobe_foobar.csv) ===")
print(df_stichprobe[["nr", "treffer", "klasse", "suchen_unter"]].to_string(index=False))



In [ ]:
# =============================================================================
# Zelle 05 – Unabhängige Referenz: foobar-Pfade vs. Index vs. Crate; Encoding
# =============================================================================
# Zweck: Die von foobar selbst geschriebenen Pfade (Stichprobe aus Zelle 04)
#   dienen als unabhängige Referenz für die Trefferstufen 1/2. Zusätzlich:
#   Wie kodiert foobar Umlaute in .m3u und .m3u8 (Bytes, BOM, Zeilenende)?
# Stufe-4-Fälle: Ergebnis der manuellen Suche als Pfad oder "fehlt".
# Ergebnis: erste Gold-Labels (data/gold, privat) – Ziel-Datei oder "fehlt".
# Zuordnung foobar-Eintrag -> Stichprobe über NFC-gleichen Referenzpfad,
#   nicht über die Reihenfolge (robust gegen Einfügefehler).
# =============================================================================

GOLD = PROJECT_ROOT / "data" / "gold"
# Manuelle Suche der Stufe-4-Fälle (Pfad relativ zur Stick-Wurzel oder "fehlt")
HANDSUCHE = {
    11: "fehlt",
    12: "Musik/2Raumwohnung 2008 08 06/_ERIK SET/set 2010 01 gut.mp3",
    13: "fehlt",
}


def lese_playlist(pfad: Path) -> dict:
    """Liest eine foobar-Playlist byte-genau: BOM, Zeilenende, Encoding, Pfade."""
    roh = pfad.read_bytes()
    bom = roh.startswith(b"\xef\xbb\xbf")
    try:
        text, encoding = roh.decode("utf-8-sig"), "utf-8"
    except UnicodeDecodeError:
        text, encoding = roh.decode("cp1252"), "cp1252 (ANSI)"
    zeilen = text.splitlines()
    return {"roh": roh, "bom": bom, "encoding": encoding,
            "crlf": b"\r\n" in roh,
            "kommentare": [z for z in zeilen if z.startswith("#")],
            "pfade": [z for z in zeilen if z.strip() and not z.startswith("#")]}


def zu_relativ(windows_pfad: str) -> str:
    """'D:\\Musik\\A\\x.mp3' -> 'Musik/A/x.mp3' (Index-Form)."""
    return windows_pfad.split(":\\", 1)[-1].replace("\\", "/")


nfc = lambda s: unicodedata.normalize("NFC", s)

# --- Referenzpfad je Stichprobe: Index-Partner (Stufe 1/2) bzw. Handsuche (Stufe 4)
df_stichprobe = pd.read_csv(GOLD / "02_04_stichprobe_foobar.csv")
df_stichprobe["referenz_pfad"] = df_stichprobe["index_pfad"]
for nr, ergebnis in HANDSUCHE.items():
    if ergebnis != "fehlt":
        df_stichprobe.loc[df_stichprobe["nr"] == nr, "referenz_pfad"] = ergebnis

for endung in ["m3u", "m3u8"]:
    pl = lese_playlist(GOLD / f"02_05_foobar_stichprobe_asus.{endung}")
    print(f"=== foobar .{endung} ===")
    print(f"  BOM: {pl['bom']} | Encoding (erkannt): {pl['encoding']} | CRLF: {pl['crlf']}")
    print(f"  Kommentar-/Kopfzeilen: {pl['kommentare'][:3]}")
    print(f"  Einträge: {len(pl['pfade'])}")

    # Byte-Evidenz: Wie stehen Umlaute in der Datei?
    for wort in ["Väth", "Planisphäre"]:
        for kodierung, b in [("UTF-8", wort.encode("utf-8")), ("cp1252", wort.encode("cp1252"))]:
            if b in pl["roh"]:
                print(f"  '{wort}' als {kodierung}-Bytes gefunden: {b.hex(' ')}")

    # Vergleich je foobar-Eintrag
    zeilen = []
    for fp in pl["pfade"]:
        rel = zu_relativ(fp)
        treffer = df_stichprobe[df_stichprobe["referenz_pfad"].fillna("").map(nfc) == nfc(rel)]
        z = treffer.iloc[0] if len(treffer) else None
        zeilen.append({
            "nr": int(z["nr"]) if z is not None else None,
            "foobar_normalform": "NFD" if nfc(rel) != rel else "NFC",
            "gleich_referenz_exakt": z is not None and rel == z["referenz_pfad"],
            "gleich_crate_exakt": z is not None and rel == z["pfad"],
            "gleich_crate_nach_nfc": z is not None and nfc(rel) == nfc(z["pfad"]),
        })
    df_vergleich = pd.DataFrame(zeilen)
    print(df_vergleich.to_string(index=False))
    print()

# --- Gold-Labels der Stichprobe ------------------------------------------------
df_gold_stichprobe = df_stichprobe[["nr", "treffer", "klasse", "pfad", "referenz_pfad"]].copy()
df_gold_stichprobe["gold_ziel"] = df_gold_stichprobe["referenz_pfad"].fillna("fehlt")
df_gold_stichprobe["gold_quelle"] = ["Handsuche" if n in HANDSUCHE else "foobar"
                                     for n in df_gold_stichprobe["nr"]]
df_gold_stichprobe.to_csv(GOLD / "02_05_gold_stichprobe.csv", index=False, encoding="utf-8")
print(df_gold_stichprobe[["nr", "treffer", "gold_quelle", "gold_ziel"]].to_string(index=False))

### Erkenntnisse Zellen 02–05: Crates, Zeichenkodierung, unabhängige Referenz

**Evidenz**

| Prüfung | Ergebnis |
|---|---|
| Integrität | 167/167 Crates vollständig bis zum letzten Byte gelesen, 0 Fehler, 1 Version |
| Umfang | 7.915 Einträge, 2.754 eindeutige Pfade (Ø 2,9 Crates je Pfad); 52,6 % der 5.236 Audiodateien haben Crate-Bezug |
| Verteilung | Tracks je Crate: Median 13, Mittelwert 47,4, Max 1.330 (stark rechtsschief) |
| pandas 3 | Datentyp `str`; NFD-Zählung Python = pandas = 656 → Normalform bleibt erhalten |
| Trefferstufen (alle 2.754 Pfade) | 1 exakt 2.075 (75,3 %) · 2 erst nach NFC 275 (10,0 %) · 3 erst ohne Akzente 0 · 4 kein Treffer 404 (14,7 %) |
| Evidenz je Stufe (Zelle 04) | Stufe 1: 3/3 zeichen-identisch, auch mit Umlaut in NFC; Stufe 2: 3/3 mit U+0308 (Crate) vs. U+00E4/U+00FC (Stick) |
| Unabhängige Referenz (Zelle 05) | foobar-Pfade 11/11 identisch mit Referenzpfad; NFD-Fälle 3/3 erst nach NFC gleich; verschobener Fall 1/1 weder exakt noch nach NFC gleich |
| Kodierung foobar | `.m3u`: cp1252, ohne BOM, `ä` = `e4` · `.m3u8`: UTF-8 mit BOM, Kopfzeile `#`, `ä` = `c3 a4` · foobar schreibt NFC |
| Gold-Labels | 13 Stichproben: 10 per foobar bestätigt, 1 verschoben (per Handsuche gefunden), 2 fehlend |

**Aussagen (aus der Evidenz abgeleitet)**

- 85,3 % der eindeutigen Crate-Pfade sind mit den Stufen 1 und 2 deterministisch auflösbar.
- NFD betrifft 300 Pfade in 72 von 167 Crates; Stufe 2 (NFC-Normalisierung beider Seiten) ist zwingend.
- Unter Pfaden, die bis auf Akzente identisch sind, gibt es keinen verlorenen Umlaut (Stufe 3 = 0).
- Die 404 Fehltreffer sind überwiegend kein Kodierungsproblem: Fehlquote ASCII 18,8 % gegenüber NFC 7,5 % und NFD 8,3 %.
- Das Ausgabeformat bestimmt die Kodierung zwingend (`.m3u` = cp1252, `.m3u8` = UTF-8).

**Grenzen und offene Punkte**

- 11 foobar-Vergleiche belegen den Mechanismus, nicht die Fehlerfreiheit (Rule of Three: Fehlerrate < 27 %).
- Nicht erfasst: verlorene Umlaute in zusätzlich verschobenen Pfaden, Mojibake-Dateinamen (20 auf dem Stick) → Zelle 06.
- 25 NFD-Pfade ohne Treffer auch nach NFC → Ursache in Zelle 06.
- Ursache der Mischung NFC/NFD in den Crates offen (alle Crates am Mac erstellt).

**Entscheidungen**

- Stufe 4.2 = NFC-Normalisierung beider Seiten.
- Ausgabeformat (`.m3u` ANSI vs. `.m3u8` UTF-8) wird in Phase 3 datenbasiert entschieden.

In [ ]:
# =============================================================================
# Zelle 06 – Fehltreffer (Stufe 4) nach Namensabgleich im Index
# =============================================================================
# Rechenvorschrift: Schlüssel k(p) = Kleinschreibung(NFC(Dateiname(p))).
#   I = alle echten Audiodateien im Index (Endung laut config, ohne AppleDouble).
#   Je Crate-Pfad p ohne Treffer in Stufe 1-3, erste zutreffende Kategorie:
#   a |{q in I : k(q) = k(p)}| = 1          (Name genau 1x im Index, anderer Ordner)
#   b |{q in I : k(q) = k(p)}| > 1          (Name mehrfach im Index)
#   c k(p) = k(Mojibake-Reparatur(q)) für q in I  (Name erst nach Reparatur)
#   d Name nicht in I, Ordner(p) im Index vorhanden
#   e Name nicht in I, Ordner(p) im Index nicht vorhanden
#   Zusatzmerkmal: Ordner(p) fehlt, aber ein Ordner gleichen Namens existiert anderswo.
# Grenze: Namensgleichheit belegt KEINE Inhaltsgleichheit (Audio). Inhaltsbeleg
#   erst über Metadaten (z. B. Länge aus Serato database V2, Zelle 07).
# Mögliche Ursachen (Hypothesen, nicht geprüft): a/b verschoben; d/e gelöscht,
#   umbenannt oder umbenannt und verschoben.
# Validierung: Abgleich mit Gold-Labels der Handsuche (Zelle 05).
# Datenschutz: Zählstatistik public; Einzelpfade privat in reports/errors.
# =============================================================================

KAT_A = "a Name 1x im Index, anderer Ordner"
KAT_B = "b Name mehrfach im Index"
KAT_C = "c Name erst nach Mojibake-Reparatur"
KAT_D = "d Name nicht im Index, Ordner vorhanden"
KAT_E = "e Name nicht im Index, Ordner nicht vorhanden"


def schluessel(s: str) -> str:
    return unicodedata.normalize("NFC", s).lower()


def mojibake_reparieren(s: str) -> str:
    """'FlÃ¼gel' -> 'Flügel': als cp1252 kodieren, als UTF-8 lesen; sonst unverändert."""
    try:
        return s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


dateiname = lambda p: p.rsplit("/", 1)[-1]
ordner = lambda p: p.rsplit("/", 1)[0]

# --- Nachschlagetabellen aus dem Index -------------------------------------------
audio = set(cfg["index"]["audio_endungen"])
df_audio = df_index[df_index["endung"].isin(audio) & ~df_index["appledouble"]].copy()
df_audio["name_key"] = df_audio["pfad_relativ"].map(lambda p: schluessel(dateiname(p)))
df_audio["name_key_rep"] = df_audio["pfad_relativ"].map(
    lambda p: schluessel(mojibake_reparieren(dateiname(p))))

name_zu_pfaden = df_audio.groupby("name_key")["pfad_relativ"].apply(list).to_dict()
moji_zu_pfaden = (df_audio[df_audio["name_key_rep"] != df_audio["name_key"]]
                  .groupby("name_key_rep")["pfad_relativ"].apply(list).to_dict())
ordner_im_index = {schluessel(ordner(p)) for p in df_index["pfad_relativ"]}
ordnernamen_im_index = {schluessel(dateiname(ordner(p))) for p in df_index["pfad_relativ"]}

print(f"Audiodateien im Index (I): {len(df_audio)} | "
      f"davon mit reparierbarem Mojibake-Namen: {(df_audio['name_key_rep'] != df_audio['name_key']).sum()}")

# --- Kategorisierung ---------------------------------------------------------------
df_rest = df_pfade[df_pfade["treffer"] == "4 kein Treffer"].copy()


def kategorisiere(p: str) -> tuple[str, list[str]]:
    k = schluessel(dateiname(p))
    if k in name_zu_pfaden:
        kand = name_zu_pfaden[k]
        return (KAT_A if len(kand) == 1 else KAT_B), kand
    if k in moji_zu_pfaden:
        return KAT_C, moji_zu_pfaden[k]
    return (KAT_D if schluessel(ordner(p)) in ordner_im_index else KAT_E), []


erg = df_rest["pfad"].map(kategorisiere)
df_rest["kategorie"] = erg.map(lambda t: t[0])
df_rest["kandidaten"] = erg.map(lambda t: t[1])
df_rest["n_kandidaten"] = df_rest["kandidaten"].map(len)
df_rest["ordnername_anderswo"] = [
    schluessel(ordner(p)) not in ordner_im_index
    and schluessel(dateiname(ordner(p))) in ordnernamen_im_index
    for p in df_rest["pfad"]]

# --- Ergebnis 1: Kategorie x Kodierungsklasse ---------------------------------------
df_kat_klasse = pd.crosstab(df_rest["kategorie"], df_rest["klasse"],
                            margins=True, margins_name="Summe")
print("\n=== Fehltreffer (eindeutige Pfade) nach Kategorie und Kodierungsklasse ===")
print(df_kat_klasse.to_string())
tabelle_speichern(df_kat_klasse.reset_index(), "02_06_fehltreffer_kategorie_klasse")

# --- Ergebnis 2: Zusatzmerkmal "Ordnername anderswo" je Kategorie --------------------
df_kat_ordner = pd.crosstab(df_rest["kategorie"], df_rest["ordnername_anderswo"],
                            margins=True, margins_name="Summe")
print("\n=== Ordner am Crate-Ort fehlt, gleichnamiger Ordner existiert anderswo (True) ===")
print(df_kat_ordner.to_string())
tabelle_speichern(df_kat_ordner.reset_index(), "02_06_fehltreffer_ordnername_anderswo")

df_rest.to_csv(ERR / "02_06_fehltreffer_kategorien.csv", index=False, encoding="utf-8")  # privat

# --- Evidenz: je Kategorie 2 Beispiele -------------------------------------------
print("\n=== Beispiele je Kategorie ===")
for kat, g in df_rest.groupby("kategorie"):
    print(f"--- {kat}: {len(g)} ---")
    for _, z in g.sample(min(2, len(g)), random_state=SEED).iterrows():
        print(f"  Crate     : {z['pfad']!r}")
        print(f"  Kandidaten: {z['kandidaten'][:3]}{' …' if z['n_kandidaten'] > 3 else ''}\n")

# --- Validierung gegen Gold-Labels der Handsuche (Zelle 05) ----------------------
df_gold = pd.read_csv(GOLD / "02_05_gold_stichprobe.csv")
print("=== Validierung gegen Handsuche (Gold) ===")
for _, g in df_gold[df_gold["gold_quelle"] == "Handsuche"].iterrows():
    z = df_rest[df_rest["pfad"] == g["pfad"]].iloc[0]
    code_ziel = (z["kandidaten"][0] if z["n_kandidaten"] == 1
                 else "fehlt" if z["n_kandidaten"] == 0 else "mehrere Kandidaten")
    print(f"  Nr. {int(g['nr'])}: Code = {z['kategorie']} -> {code_ziel!r}")
    print(f"          Gold = {g['gold_ziel']!r} | übereinstimmend: {code_ziel == g['gold_ziel']}")

# --- Grafik --------------------------------------------------------------------------
df_plot_k = df_rest["kategorie"].value_counts().sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.barh(df_plot_k.index, df_plot_k.values, color=FARBEN["gold"])
for y, v in enumerate(df_plot_k.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=12)
ax.set_xlabel("eindeutige Crate-Pfade ohne Treffer in Stufe 1–3")
ax.set_title(f"{len(df_rest)} Fehltreffer nach Namensabgleich im Index "
             "(Namensgleichheit ≠ Inhaltsgleichheit)")
figur_speichern(fig, "02_06_fehltreffer_kategorien", FIG)
plt.show()

### Erkenntnisse Zelle 06: Fehltreffer nach Namensabgleich

**Evidenz** (404 eindeutige Crate-Pfade ohne Treffer in Stufe 1–3; Vergleich über Kleinschreibung(NFC(Dateiname)) gegen 5.236 Audiodateien)

| Kategorie | Pfade | Anteil an 404 | Anteil an 2.754 |
|---|---|---|---|
| a Name 1x im Index, anderer Ordner | 166 | 41,1 % | 6,0 % |
| b Name mehrfach im Index | 0 | 0 % | 0 % |
| c Name erst nach Mojibake-Reparatur | 0 | 0 % | 0 % |
| d Name nicht im Index, Ordner vorhanden | 8 | 2,0 % | 0,3 % |
| e Name nicht im Index, Ordner nicht vorhanden | 230 | 56,9 % | 8,4 % |

- Zusatzmerkmal: 25 der 166 a-Pfade (15,1 %) haben einen gleichnamigen Ordner an anderer Stelle; in d und e: 0.
- NFD: alle 25 NFD-Fehltreffer liegen in e; keine Überschneidung mit den 25 Fällen „Ordnername anderswo“.
- Mojibake: 1 reparierbarer Audio-Dateiname im Index, kein Crate-Verweis darauf.
- Validierung gegen Handsuche: 3/3 übereinstimmend (Nr. 12 mit identischem Zielpfad, Nr. 11 und 13 ohne Kandidat).

**Aussagen**

- Über Namensgleichheit liegt für 2.516 von 2.754 Pfaden (91,4 %) ein Kandidat vor (Stufe 1: 2.075, Stufe 2: 275, Kategorie a: 166).
- Für 238 Pfade (8,6 %) liefert der Dateiname keinen Kandidaten.
- Mehrdeutigkeit (b) und Mojibake (c) treten bei den Fehltreffern nicht auf.

**Grenzen**

- Namensgleichheit belegt keine Inhaltsgleichheit; Kategorie a ist ein Kandidat, nicht „sicher“.
- Ursachen sind nicht geprüft; Hypothesen: a verschoben (25 Fälle mit verschobenem Ordner, Beispiel `_ERIK SET`, laut Projektinhaber versehentlich verschoben); d/e gelöscht, umbenannt oder umbenannt und verschoben.

**Konsequenz**

- Inhaltsbeleg und Suche ohne Dateinamen erfordern Metadaten → Serato `database V2` (Zelle 07).
- Fall `_ERIK SET` = erster bestätigter Reparaturvorschlag für den Pflichtbericht (Ausführung durch den Nutzer, P1/P4).